# 01 – Thống kê dữ liệu (W1-07) và kiểm tra lỗi dữ liệu (W1-08)

Hướng dẫn từng bước: `docs/huong_dan/W1-07.md`, `docs/huong_dan/W1-08.md`.
Ô nào có chữ **TODO** là phần m tự viết, mấy ô khác t viết sẵn rồi, m cứ bấm chạy lần lượt từ trên xuống.
Hình vẽ lưu vào `results/figures/eda_*.png` (nằm trong Drive chung) để dán vào báo cáo mục 2.

In [ ]:
# Ô 0 – chuẩn bị (lần nào mở notebook cũng chạy ô này trước; Colab hỏi quyền vào Drive thì bấm cho phép)
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/NLI'      # lối tắt tới Drive chung (xem docs/huong_dan/README.md, mục đầu)
!git clone -q https://github.com/quanuiter/NLI.git /content/NLI 2>/dev/null || git -C /content/NLI pull -q
%cd /content/NLI
!pip install -q -r requirements.txt
import os
for sub in ['data/processed', 'tools', 'results']:
    target = os.path.join(DRIVE_DIR, sub); os.makedirs(target, exist_ok=True)
    if os.path.islink(sub): continue
    if os.path.isdir(sub):
        !cp -rn {sub}/. {target}/ && rm -rf {sub}
    os.symlink(target, sub)

In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt
from src.data import DATASET_SPLITS, LABELS, load_split
from src.embeddings import tokenize

# Gom hết dữ liệu vào 1 bảng tên df (phải ra 7 tập, tổng 119.776 dòng)
df = pd.concat([load_split(d, s) for d, splits in DATASET_SPLITS.items() for s in splits], ignore_index=True)
df['tap'] = df['source'] + '_' + df['split']
df.groupby('tap').size()

## 1. Mỗi nhãn có bao nhiêu câu

In [ ]:
pd.crosstab(df['tap'], df['label']).reindex(columns=LABELS)

**TODO 1:** đổi bảng trên sang % (gợi ý: thêm `normalize='index'` vào `pd.crosstab(...)`), rồi viết 1–2 câu nhận xét:
bộ nào 3 nhãn đều nhau, bộ nào bị lệch về nhãn nào.

In [ ]:
# TODO 1

## 2. Câu dài bao nhiêu (đếm số âm tiết)

In [ ]:
df['len_p'] = df['premise'].map(lambda s: len(tokenize(s)))
df['len_h'] = df['hypothesis'].map(lambda s: len(tokenize(s)))
df.groupby('tap')[['len_p', 'len_h']].agg(['mean', 'median', 'max']).round(1)

**TODO 2:** vẽ biểu đồ độ dài câu 2 (hypothesis) của 3 tập `xnli_train`, `vianli_train`, `vinli_test` chung 1 hình,
lưu `results/figures/eda_do_dai.png`. Code mẫu có trong `docs/huong_dan/W1-07.md`, m copy vào là chạy.
Nhận xét: bộ nào câu dài hơn? Câu 2 của mỗi nhãn dài khác nhau không? (xem bằng `df.groupby(['tap','label'])['len_h'].mean()`)

In [ ]:
# TODO 2

## 3. Hai câu trùng nhau bao nhiêu từ (%)

In [ ]:
def jaccard(p, h):
    a, b = set(t.lower() for t in tokenize(p)), set(t.lower() for t in tokenize(h))
    return len(a & b) / max(1, len(a | b))

df['jaccard'] = [jaccard(p, h) for p, h in zip(df['premise'], df['hypothesis'])]
(df.pivot_table(index='label', columns='tap', values='jaccard') * 100).round(1).reindex(LABELS)

**TODO 3:** vẽ biểu đồ cột cho bảng trên (gợi ý: `bang.plot.bar()` rồi `plt.savefig('results/figures/eda_jaccard.png', dpi=150, bbox_inches='tight')`).
Nhận xét: bộ nào câu entailment trùng từ nhiều nhất? ViANLI có ngược lại không? (Số phải khớp `docs/ket_qua_m2.md` mục 4.)

In [ ]:
# TODO 3

## 4. Từ “không” hay nằm ở nhãn nào

In [ ]:
NEG = r'\b(không|chưa|chẳng|chả|đâu có|không hề|chưa từng)\b'
# TODO 4: 1) tạo cột đánh dấu câu 2 có từ phủ định:  df['h_neg'] = df['hypothesis'].str.contains(NEG, case=False)
#         2) tính % theo nhãn và theo tập:            df.pivot_table(index='label', columns='tap', values='h_neg')
#         Câu hỏi: ở XNLI, từ phủ định hay đi với nhãn nào? ViNLI, ViANLI có giống không?
#         (bài ViANLI mục 7.6 nói: ở ViNLI "không" lại hay đi với entailment)

---
# Phần W1-08: kiểm tra lỗi dữ liệu

In [ ]:
# 5. Câu rỗng hoặc quá ngắn (câu kiểu "Đúng vậy." vẫn là câu bình thường, không phải lỗi)
df[(df['len_p'] < 3) | (df['len_h'] < 2)][['tap', 'premise', 'hypothesis', 'label']].head(20)

In [ ]:
# 6. Cặp câu bị trùng trong cùng một tập
dup = df[df.duplicated(['tap', 'premise', 'hypothesis'], keep=False)]
print(dup.groupby('tap').size())
# TODO 6: trong mấy cặp trùng, có cặp nào giống hệt nhau mà KHÁC nhãn không? (code mẫu trong docs/huong_dan/W1-08.md)

In [ ]:
# 7. Câu trong tập thi có bị lẫn trong phần học không?
def leak(train_tap, test_tap):
    tr = set(zip(df.loc[df.tap == train_tap, 'premise'], df.loc[df.tap == train_tap, 'hypothesis']))
    te = df[df.tap == test_tap]
    n = sum((p, h) in tr for p, h in zip(te['premise'], te['hypothesis']))
    return n, len(te)

for a, b in [('xnli_train', 'xnli_test'), ('vianli_train', 'vianli_test'), ('vianli_train', 'vinli_test')]:
    print(a, '->', b, leak(a, b))
# TODO 7: làm lại y vậy nhưng chỉ so câu 1 (premise), ghi lại số lượng

**TODO 8 – tổng kết:** viết bảng *loại lỗi · bao nhiêu · xử lý thế nào*. T điền sẵn mấy lỗi t đã sửa trong `docs/huong_dan/W1-08.md`,
m copy qua rồi thêm kết quả ô 5–7 của m. Bảng này đưa vào báo cáo mục 2.